## **Execução de RPC + Ollama no Google Colab**  

Este tutorial apresenta um guia detalhado para a configuração e execução de **RPC + Ollama** em um ambiente **Google Colab**.  

### **Requisitos**  

Antes de iniciar, certifique-se de atender aos seguintes requisitos:  

- Conta no **Ngrok** com um **token de autenticação** válido.  
- Acesso a um ambiente **Google Colab** com suporte a **GPU Nvidia (T4 - free tier)**.  
- **Python** instalado na máquina local.  

Os próximos passos detalham o processo de instalação e configuração, garantindo a correta execução do ambiente para testes e experimentação.



---



**I. Instalação das Dependências do Ollama**


> O pacote `pciutils` é necessário para que o Ollama consiga identificar corretamente o tipo de GPU disponível na instância do Colab.

> A instalação do Ollama na instância em tempo de execução será realizada pelo seguinte comando `sh curl -fsSL https://ollama.com/install.sh | sh`

In [ ]:
!sudo apt update
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

**II. Instalação das Dependências do Python**  

> O pacote `langchain-ollama` é necessário para integrar o Ollama com a biblioteca **Langchain**, facilitando a interação com modelos de linguagem.  

> O pacote `pyngrok` é necessário para configurar e gerenciar o túnel do Ngrok diretamente a partir do código Python.  

In [ ]:
!pip install langchain-ollama
!pip install langchain_community
!pip install pyngrok

**III. Autenticar Ngrok com Authtoken**
> Esta parte é essencial para comunição fora do Colab, para obter o token basta [acessar a página](https://dashboard.ngrok.com/get-started/your-authtoken), e copiar o token após se autenticar.


In [ ]:
!ngrok authtoken <authtoken>

**IV. Execução do Ollama**  

> Para utilizar o Ollama, é necessário que ele seja executado como um serviço em segundo plano, paralelo aos seus scripts. No entanto, como os Jupyter Notebooks são projetados para rodar os blocos de código de forma sequencial, isso dificulta a execução simultânea de dois blocos de código. Como solução, vamos criar um serviço utilizando o módulo `subprocess` em Python, garantindo que a execução de uma célula não bloqueie a execução das demais.

> Além disso, para garantir que o serviço Ollama esteja completamente em funcionamento antes de baixar o modelo, utilizamos um pequeno delay com o comando `time.sleep(5)`, o que dá tempo para o serviço ser inicializado corretamente.

In [ ]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

**V. Baixando o Modelo**  

> Para utilizar o modelo LLM, é necessário fazer o download utilizando o comando `ollama pull llama3.1`. Este comando irá baixar o modelo Llama 3.1 8b, que pode ser utilizado em seu ambiente Colab.  

> Se desejar utilizar outros modelos, você pode consultar a lista completa de modelos disponíveis no site oficial do Ollama: [https://ollama.com/library](https://ollama.com/library).  

In [ ]:
!ollama pull llama3.1

**VI. Iniciar servidor RPC com Ollama e RAG**
> Este código configura um servidor XML-RPC que permite interagir com o modelo de texto rodando no Ollama + Langchain.



In [ ]:
from langchain_core.prompts import PromptTemplate

# Define o template (personalidade) que será utilizada pelo modelo.
# Para respostas mais precisas o template é estruturado em inglês.
prompt_template_sql = PromptTemplate.from_template(
    """
    Given the following SQL database schema:
    {schema_info}

    Convert the following natural language query into a SQL SELECT statement using only the schema as reference:
    {input_text}

    Rules:
    1. If the user asks for anything that is not relevant to queries, just reply: "I don't know."
    2. Ignore unrecognized tables/columns, don’t try to create new ones.
    3. Return only SQL queries in ``` marks, and don’t leave notes.
    4. Use SELECT statements only.
    5. Table names are duplicated due to Django; for example, usuario should be usuario_usuario.
    6. Most tables/columns are in Brazilian Portuguese, natural language, and may contain accents. You will need to remove them to correctly find them in the context.
    """
)

In [ ]:
# Define a função que gera consultas SQL
def gerar_resposta(prompt_usuario: str) -> str:
    print('------ NOVA REQUISIÇÃO DO USUÁRIO ------')
    print('PERGUNTA > ', prompt_usuario)
    try:
        # Generate and process response
        llm_response = sql_chain.invoke({
            "input_text"  : prompt_usuario,
            "schema_info" : json.dumps(esquema_banco)
        })
        print('RESPOSTA > ', llm_response)

        # Extract SQL query
        sql_query = extrair_consulta_sql(llm_response)
        print('QUERYSQL > ', sql_query)

        return sql_query
    except Exception as e:
        error_msg = f"Erro ao gerar resposta: {str(e)}"
        print(error_msg)
        return error_msg
    finally:
        print('------- FIM REQUISIÇÃO DO USUÁRIO ------')

In [ ]:
def extrair_consulta_sql(resposta_llm: str) -> str:
    """
    Extrai e retorna uma consulta SQL formatada a partir da resposta do LLM,
    que deve estar entre crases triplas.
    Retorna uma mensagem de erro se nenhuma consulta SQL válida for encontrada.
    """
    try:
        # Padrão regex para encontrar conteúdo entre ```sql``` ou apenas ```
        padrao = r'```(?:sql)?\s*(.*?)\s*```'
        match = re.search(padrao, resposta_llm, re.DOTALL)

        if match:
            # Extrai a consulta SQL e remove espaços em branco
            consulta_sql = match.group(1).strip()

            # Remove o prefixo 'SQL' se existir
            if consulta_sql.upper().startswith('SQL'):
                consulta_sql = consulta_sql[3:].strip()

            # Verifica se a consulta começa com SELECT
            if not consulta_sql.upper().startswith('SELECT'):
                return "Erro: A consulta retornado pelo servidor não é para recuperar dados!"

            # Remove ponto e vírgula final, se houver
            return consulta_sql.rstrip(';')

        return "Erro: Não foi possível gerar uma consulta com essas informações, por favor, tente novamente."

    except Exception as erro:
        return f"Erro: Falha ao processar a consulta SQL - {str(erro)}"

In [ ]:
from langchain_ollama import OllamaLLM

from pyngrok import ngrok

from xmlrpc.server import SimpleXMLRPCServer
from xmlrpc.server import SimpleXMLRPCRequestHandler

import json
import re

# Recupera o contexto para utlizar no RAG via arquivo JSON.
with open("esquema_banco.json", "r") as file:
    esquema_banco = json.load(file)
    print(esquema_banco)

# Incializa o modelo usando recém criado llama3.1.
llm = OllamaLLM(
    model="llama3.1",
    temperature=0.1  # Menor tempereatura = Respota mais determinisitca.
)

# Inicializa a chain para fazer chamadas para LLM usando o template.
sql_chain = prompt_template_sql | llm

# Cria o serivdor RPC para responder requisições.
class ManipuladorDeRequisicoes(SimpleXMLRPCRequestHandler):
    rpc_paths = ("/RPC2",)

server = SimpleXMLRPCServer(
    ("0.0.0.0", 1334),
    requestHandler=ManipuladorDeRequisicoes,
    allow_none=True
)

# Registra somente a função de gerar consultas SQL para o servidor.
server.register_function(gerar_resposta, "gerar_resposta")

# Cria uma URL pública pelo ngrok para acessar o serviço fora do Google Colab.
try:
    url_publica = ngrok.connect(1334, bind_tls=True).public_url
    print("URL Pública: ", url_publica)
except Exception as e:
    print(f"Falha ao conectart com ngrok: {str(e)}")
    raise

# Instancia o servidor e executa sem interrupção
print("Servidor XML-RPC em execução...")
server.serve_forever()
